# Measure analysis

##### Imports and params

In [ ]:
from __future__ import annotations


from ast import literal_eval
from pathlib import Path
import os
import re
from typing import Dict, List, Tuple

import ipywidgets as widgets
import numpy as np
import pandas as pd
import pprint

import re
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import matplotlib.patheffects as pe

In [ ]:
plt.rcParams.update({
"figure.figsize": (18, 10),
"axes.grid": True,
"grid.linestyle": ":",
"grid.alpha": 0.5,
"axes.titlesize": 14,
"axes.labelsize": 12,
"legend.fontsize": 10,
})

In [ ]:
PATH_RESULTS = Path("./measureresults_per_plant.csv")
PATH_TB_LOGS = Path("~/Documents/code/RSMLExtraction/Results/Training/Logs/").expanduser()

## Loading database

In [ ]:
df_plant_wid = pd.read_csv(PATH_RESULTS)
df_plant_wid["root_ids"] = df_plant_wid["root_ids"].apply(literal_eval)
# remove metric = apls
#df_plant_wid = df_plant_wid[df_plant_wid['metric'] != 'apls'].reset_index(drop=True)

In [ ]:

dict_last_epochs = {}
dict_last_epochs['Unet_dice'] = 261
dict_last_epochs['Unet_cldice_dice'] = 260
dict_last_epochs['Unet_bce_dice'] = 267
dict_last_epochs['Unet_bce'] = 263
dict_last_epochs['Segformer_dice'] = 300
dict_last_epochs['Segformer_bce_dice'] = 285
dict_last_epochs['Segformer_bce'] = 286

In [ ]:
# if model name has no numeric suffix, replace model name by model_name_last where last is taken from dict_last_epochs
def add_epoch_suffix(m):
    tail = m.split('_')[-1]
    if tail.isdigit():
        return m
    return f"{m}_{dict_last_epochs.get(m, tail)}"

df_plant_wid["model"] = df_plant_wid["model"].map(add_epoch_suffix)

In [ ]:
# order by model name and epoch number
df_plant_wide = df_plant_wid.sort_values(by=['model', 'split', 'box', 'time', 'root_ids', 'metric']).reset_index(drop=True)
df_plant_wide

In [ ]:
_df_grouped = (
df_plant_wid
.set_index(["model", "split", "box", "metric", "root_ids", "source", "time"]).sort_index(level="time")
)

MODELS: List[str] = _df_grouped.index.get_level_values("model").unique().tolist()
SPLITS: List[str] = _df_grouped.index.get_level_values("split").unique().tolist()
BOXES: List[str] = _df_grouped.index.get_level_values("box").unique().tolist()
METRICS: List[str] = _df_grouped.index.get_level_values("metric").unique().tolist()
SOURCES: List[str] = _df_grouped.index.get_level_values("source").unique().tolist()

pprint.pprint({
    "Models": MODELS,
    "Splits": SPLITS,
    "Boxes": BOXES,
    "Metrics": METRICS,
    "Sources": SOURCES
})

### Helpers de sélection

In [ ]:
SPLIT_TO_BOXES = {
    SPLITS[0]: BOXES[0:5],  # 0 à 4
    SPLITS[1]: BOXES[5:10], # 5 à 9
}
pprint.pprint(SPLIT_TO_BOXES)

del BOXES 
del SPLITS

In [ ]:
DefLevels = Tuple[str, str, str, str]

def get_root_ids(model: str, split: str, box: str, metric: str) -> List[Tuple[int, ...]]:
    return (
    _df_grouped
    .xs((model, split, box, metric), level=("model", "split", "box", "metric"))
    .index.get_level_values("root_ids").unique().tolist()
    )


def slice_df(model: str, split: str, box: str, metric: str, root: Tuple[int, ...]) -> pd.DataFrame:
    return (
    _df_grouped
    .xs((model, split, box, metric, root), level=("model", "split", "box", "metric", "root_ids"))
    .reset_index()
    .assign(time=lambda d: pd.to_numeric(d["time"], errors="coerce").astype(float))
    .sort_values("time")
    )

In [ ]:
model = MODELS[10]
split = list(SPLIT_TO_BOXES.keys())[0]
box = list(SPLIT_TO_BOXES[split])[4]
metric = METRICS[0]
root_ids = get_root_ids(model, split, box, metric)
print(root_ids)
root = root_ids[0]
sliced_df = slice_df(model, split, box, metric, root)

pprint.pprint({
    "model": model,
    "split": split,
    "box": box,
    "metric": metric,
    "root": root,
    "root_ids": root_ids,
    "sliced_df": sliced_df
})

In [ ]:
STYLE_MAP = {
"before_expertized": dict(color="tab:orange", linestyle="--", marker=None, linewidth=2.2, markersize=7, alpha=0.8, xoff=-0.0),
"expertized": dict(color="tab:green", linestyle="-", marker="o", linewidth=2.4, markersize=7, alpha=0.9, xoff=0.00),
"Prediction": dict(color="tab:red", linestyle="-", marker="x", linewidth=2.2, markersize=7, alpha=0.9, xoff=+0.00),
}

def plot_root(model: str, split: str, box_idx: int, metric: str, root_idx: int):
    boxes_by_split = SPLIT_TO_BOXES.get(split, [])
    # Exemple original utilisait: box = boxes[box_idx + split_idx * 5]
    box = boxes_by_split[box_idx]


    roots = get_root_ids(model, split, box, metric)
    if len(roots) == 0:
        print("Aucune racine pour cette combinaison.")
        return
    root = roots[min(max(root_idx, 0), len(roots)-1)]


    df_tmp = slice_df(model, split, box, metric, root)


    plt.figure(figsize=(16, 8))
    for source, style in STYLE_MAP.items():
        d = df_tmp[df_tmp["source"] == source]
        if d.empty:
            continue
        x = d["time"]
        plt.plot(
        x, d["value"],
        label=source,
        color=style["color"], linestyle=style["linestyle"], marker=style["marker"],
        linewidth=style["linewidth"], markersize=style["markersize"], alpha=style["alpha"],
        )
        # Annotation du dernier point
        plt.text(x.iloc[-1], d["value"].iloc[-1], f" {source}", va="center", fontsize=9, color=style["color"])


    xt = sorted(df_tmp["time"].unique())
    plt.xticks(xt, xt)
    plt.title(f"{metric} — {model} / {split} / {box} / root {root}")
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.legend(title="Source")
    plt.tight_layout()
    plt.show()


w_model = widgets.Dropdown(options=MODELS, value=MODELS[0], description="Model")
w_split = widgets.Dropdown(options=list(SPLIT_TO_BOXES.keys()), value=list(SPLIT_TO_BOXES.keys())[0], description="Split")
w_box = widgets.IntSlider(min=0, max=4, step=1, value=0, description="Box idx") 
w_metric= widgets.Dropdown(options=METRICS, value=METRICS[0], description="Metric")
w_root = widgets.IntSlider(min=0, max=len(get_root_ids(w_model.value, w_split.value, list(SPLIT_TO_BOXES[w_split.value])[w_box.value], w_metric.value)) - 1, step=1, value=0, description="Root idx")

In [ ]:
def _update_root_slider(*_):
    try:
        roots = get_root_ids(w_model.value, w_split.value,
                             SPLIT_TO_BOXES[w_split.value][w_box.value], w_metric.value)
    except Exception:
        roots = []
    w_root.max = max(len(roots) - 1, 0)
    if w_root.value > w_root.max:
        w_root.value = w_root.max


for wid in (w_model, w_split, w_box, w_metric):
    wid.observe(_update_root_slider, "value")
_update_root_slider()


widgets.VBox([
    widgets.HBox([w_model, w_split, w_box, w_metric, w_root]),
    widgets.interactive_output(
        plot_root,
        dict(model=w_model, split=w_split, box_idx=w_box,
             metric=w_metric, root_idx=w_root)
    )
])

## Error abs and signed

In [ ]:
DfPivot = (
    df_plant_wid
    .pivot_table(index=["model", "metric", "time", "split", "box", "root_ids"], columns="source", values="value")
    .reset_index()
)
DfPivot

In [ ]:
DfError = DfPivot.copy()
DfError["abs_error"] = (DfError.get("expertized") - DfError.get("Prediction")).abs()
DfError["real_error"] = (DfError.get("expertized") - DfError.get("Prediction"))
DfError = DfError[["model", "metric", "time", "split", "box", "root_ids", "abs_error", "real_error"]]

DfError

In [ ]:
DfMeanErr = (
    DfError.groupby(["model", "metric", "time"], as_index=False) # val and test set combined - IMPORTANT NOTE
    .agg(abs_error_mean=("abs_error", "mean"), real_error_mean=("real_error", "mean"), std_abs_error=("abs_error", "std"))
)
DfMeanErr

## Helpers

In [ ]:
# -- Helpers
_epoch_re = re.compile(r'_(\d+)$')

def base_name(model: str) -> str:
    # "Unet_bce_060" -> "Unet" ; "Segformer_bce_dice_140" -> "Segformer"
    return model[:model.rfind('_')]

def extract_epochs(model: str):
    m = _epoch_re.search(model)
    return int(m.group(1)) if m else None

def blend_with_white(color, intensity: float):
    """
    intensity in [0,1]; 0 -> blanc, 1 -> couleur d'origine.
    """
    c = np.array(mcolors.to_rgb(color))
    return tuple((1 - intensity) * 1.0 + intensity * c)

# Palette VIVE (couleurs marquées)
BOLD_COLORS = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00",
    "#a65628", "#f781bf", "#999999", "#66c2a5", "#a6cee3",
    "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e",
]

ALL_MODELS_ERR = sorted(DfMeanErr["model"].unique().tolist())
print(ALL_MODELS_ERR)

# keep only : 'Segformer_bce_286', 'Segformer_bce_dice_285', 'Segformer_dice_300', 'Unet_bce_263', 'Unet_bce_dice_267', 'Unet_cldice_dice_260', 'Unet_dice_261'
ALL_MODELS_ERR = [
    'Segformer_bce_286', 'Segformer_bce_dice_285', 'Segformer_dice_300',
    'Unet_bce_263', 'Unet_bce_dice_267', 'Unet_cldice_dice_260', 'Unet_dice_261'
]

def plot_distance_all_models(metric: str, use_error_bars: bool = True):
    d = DfMeanErr[DfMeanErr["metric"] == metric].copy()
    if d.empty:
        print(f"Aucune donnée pour {metric}.")
        return

    # Prépare colonnes
    d["time"] = pd.to_numeric(d["time"], errors="coerce")
    d["base"] = d["model"].map(base_name)
    d["epochs"] = d["model"].map(extract_epochs)
    d = d.sort_values(["base", "model", "time"])

    # Teinte par famille (couleurs marquées)
    bases = d["base"].dropna().unique().tolist()
    base2color = {b: BOLD_COLORS[i % len(BOLD_COLORS)] for i, b in enumerate(bases)}

    # Intensité par #epochs, normalisée par famille
    inten_min, inten_max = 0.45, 1.00
    base2norm = {}
    for b in bases:
        e = d.loc[d["base"] == b, "epochs"].dropna().values
        if len(e) == 0:
            base2norm[b] = (None, None, None)
        else:
            e_min, e_max = np.min(e), np.max(e)
            span = max(1, e_max - e_min)
            base2norm[b] = (e_min, e_max, span)

    plt.figure(figsize=(14, 7))

    already_labeled = set()
    for m in ALL_MODELS_ERR:
        dm = d[d["model"] == m]
        if dm.empty:
            continue

        b = base_name(m)
        base_col = base2color.get(b, "#377eb8")
        ep = extract_epochs(m)

        if ep is None or base2norm[b][0] is None:
            intensity = 0.75
        else:
            e_min, _, span = base2norm[b]
            norm = (ep - e_min) / span  # 0..1
            intensity = inten_min + (inten_max - inten_min) * norm

        line_color = blend_with_white(base_col, intensity)

        label = b if b not in already_labeled else None
        if label is not None:
            already_labeled.add(b)

        # Ligne bien lisible + marqueurs contrastés
        plt.plot(
            dm["time"], dm["abs_error_mean"],
            marker="o", markersize=6, markerfacecolor="white",
            markeredgewidth=1.6, markeredgecolor=line_color,
            linewidth=2.6, solid_capstyle="round", solid_joinstyle="round",
            alpha=0.98, color=line_color, label=label, zorder=3
        )


    # Légende compacte: une par famille (teinte quasi pleine)
    legend_handles = []
    for b in bases:
        c = blend_with_white(base2color[b], 0.96)
        legend_handles.append(Line2D([0], [0], color=c, lw=4, label=b))
    plt.legend(handles=legend_handles, title="Famille de modèle", ncol=2, frameon=True)

    plt.title(f"Evolution of the number of detected organs of the plant for each model")
    plt.xlabel("Time"); plt.ylabel("Mean absolute error")
    plt.grid(True, axis="y", alpha=0.25, linewidth=0.8)
    plt.tight_layout()
    plt.show()

# Widget : vous pouvez basculer bandes/barres d'erreur en jouant avec use_error_bars
widgets.interactive(
    plot_distance_all_models,
    metric=widgets.Dropdown(options=METRICS, value=METRICS[0]),
    use_error_bars=widgets.Checkbox(value=True, description="Barres d'erreur (lisibles)")
)


In [ ]:
def plot_distance_all_models(metric: str, use_error_bars: bool = True):
    d = DfMeanErr[DfMeanErr["metric"] == metric].copy()
    if d.empty:
        print(f"Aucune donnée pour {metric}.")
        return

    # Prépare colonnes
    d["time"] = pd.to_numeric(d["time"], errors="coerce")
    d["base"] = d["model"].map(base_name)
    d["epochs"] = d["model"].map(extract_epochs)
    d = d.sort_values(["base", "model", "time"])

    # Teinte par famille (couleurs marquées)
    bases = d["base"].dropna().unique().tolist()
    base2color = {b: BOLD_COLORS[i % len(BOLD_COLORS)] for i, b in enumerate(bases)}

    # Intensité par #epochs, normalisée par famille
    inten_min, inten_max = 0.45, 1.00
    base2norm = {}
    for b in bases:
        e = d.loc[d["base"] == b, "epochs"].dropna().values
        if len(e) == 0:
            base2norm[b] = (None, None, None)
        else:
            e_min, e_max = np.min(e), np.max(e)
            span = max(1, e_max - e_min)
            base2norm[b] = (e_min, e_max, span)

    plt.figure(figsize=(14, 7))

    already_labeled = set()
    for m in ALL_MODELS_ERR:
        dm = d[d["model"] == m]
        if dm.empty:
            continue

        b = base_name(m)
        base_col = base2color.get(b, "#377eb8")
        ep = extract_epochs(m)

        if ep is None or base2norm[b][0] is None:
            intensity = 0.75
        else:
            e_min, _, span = base2norm[b]
            norm = (ep - e_min) / span  # 0..1
            intensity = inten_min + (inten_max - inten_min) * norm

        line_color = blend_with_white(base_col, intensity)

        label = b if b not in already_labeled else None
        if label is not None:
            already_labeled.add(b)

        # Ligne bien lisible + marqueurs contrastés
        plt.plot(
            dm["time"], dm["real_error_mean"],
            marker="o", markersize=6, markerfacecolor="white",
            markeredgewidth=1.6, markeredgecolor=line_color,
            linewidth=2.6, solid_capstyle="round", solid_joinstyle="round",
            alpha=0.98, color=line_color, label=label, zorder=3
        )


    # Légende compacte: une par famille (teinte quasi pleine)
    legend_handles = []
    for b in bases:
        c = blend_with_white(base2color[b], 0.96)
        legend_handles.append(Line2D([0], [0], color=c, lw=4, label=b))
    plt.legend(handles=legend_handles, title="Famille de modèle", ncol=2, frameon=True)

    plt.title(f"Evolution of the number of detected organs of the plant for each model - signed error")
    plt.xlabel("Time"); plt.ylabel("Mean absolute error")
    plt.grid(True, axis="y", alpha=0.25, linewidth=0.8)
    plt.tight_layout()
    plt.show()

# Widget : vous pouvez basculer bandes/barres d'erreur en jouant avec use_error_bars
widgets.interactive(
    plot_distance_all_models,
    metric=widgets.Dropdown(options=METRICS, value=METRICS[0]),
    use_error_bars=widgets.Checkbox(value=True, description="Barres d'erreur (lisibles)")
)


In [ ]:
def extract_last_metrics(base_path: Path) -> Dict[str, Dict[str, float]]:
    """Explore `base_path` -> {metric_name: {model_name: last_value}}"""
    metrics: Dict[str, Dict[str, float]] = {}
    for model_name in os.listdir(base_path):
        model_path = base_path / model_name / "logs" / "scalars"
        if not model_path.is_dir():
            continue
        for file in os.listdir(model_path):
            if not file.endswith(".csv"):
                continue
            fp = model_path / file
            try:
                df = pd.read_csv(fp)
                if "value" not in df.columns or df.empty:
                    continue
                last_val = float(df["value"].iloc[-1])
                metric_name = file.replace("val_", "").replace(".csv", "")
                metrics.setdefault(metric_name, {})[model_name] = last_val
            except Exception as e:
                print(f"⚠️ Lecture impossible {fp}: {e}")
    return metrics

def extract_metrics_epoch(base_path: Path, epoch_num: int) -> Dict[str, Dict[str, float]]:
    """Retourne les valeurs à l'époque `epoch_num` (indexée 1) si possible, sinon fallback last."""
    metrics: Dict[str, Dict[str, float]] = {}
    for model_name in os.listdir(base_path):
        model_path = base_path / model_name / "logs" / "scalars"
        if not model_path.is_dir():
            continue
        for file in os.listdir(model_path):
            if not file.endswith(".csv"):
                continue
            fp = model_path / file
            try:
                df = pd.read_csv(fp)
                if "value" not in df.columns or df.empty:
                    continue
                if epoch_num >= len(df):
                    print(f"⚠️ Pas assez d'époques dans {fp}")
                    return extract_last_metrics(base_path)
                step_idx = int(df["step"].iloc[epoch_num - 2])
                epoch_val = float(df["value"].iloc[step_idx])
                metric_name = file.replace("val_", "").replace(".csv", "")
                metrics.setdefault(metric_name, {})[model_name] = epoch_val
            except Exception as e:
                print(f"⚠️ Lecture impossible {fp}: {e}")
    return metrics

def extract_metrics_corresponding_epoch(base_path: Path, all_models: List[str]) -> Dict[str, Dict[str, float]]:
    """`all_models` comme ['Unet_bce_063', ...] -> aligne les époques par suffixe numérique _XYZ."""
    epochs = [int(m.split('_')[-1]) if m.split('_')[-1].isdigit() else -1 for m in all_models]
    base_names = [f"{m[:m.rfind('_')]}" for m in all_models]
    metrics: Dict[str, Dict[str, float]] = {}
    print(np.unique(base_names))

    for model_dir in os.listdir(base_path):
        model_path = base_path / model_dir / "logs" / "scalars"
        if not model_path.is_dir():
            continue
        if model_dir not in base_names:
            continue
        for ref_model, epoch_num in zip(all_models, epochs):
            if not ref_model[:ref_model.rfind('_')] == model_dir.split('/')[-1]:
                continue
            for file in os.listdir(model_path):
                if not file.endswith(".csv"):
                    continue
                fp = model_path / file
                try:
                    df = pd.read_csv(fp)
                    if "value" not in df.columns or df.empty:
                        continue
                    if epoch_num > len(df):
                        print(model_dir, model_path,ref_model, epoch_num)
                        print(f"⚠️ Pas assez d'époques dans {fp} pour epoch {epoch_num}")
                        print(df)
                        return extract_last_metrics(base_path)
                    
                    step_idx = int(df["step"].iloc[epoch_num - 2])
                    epoch_val = float(df["value"].iloc[step_idx])
                    metric_name = file.replace("val_", "").replace(".csv", "")
                    metrics.setdefault(metric_name, {})[ref_model] = epoch_val
                except Exception as e:
                    print(f"⚠️ Lecture impossible {fp}: {e}")
    return metrics

# --- Récupération valeurs CV pour les modèles présents côté graph ---
ALL_MODELS_GRAPH = sorted(DfMeanErr["model"].unique().tolist())
cv_metrics_results = extract_metrics_corresponding_epoch(PATH_TB_LOGS, all_models=ALL_MODELS_GRAPH)

In [ ]:
# Jointure tableau récap pour scatter
DfCvGraph = (
    DfMeanErr.copy() # NOTE : from DfMeanErr
    .groupby(["metric","model"])[["abs_error_mean","std_abs_error"]] # IGNORE reel error, and mean over all times (sum of of the error of each model for each metric at all times)
    .mean().reset_index()
)
for metric_name, values_by_model in cv_metrics_results.items():
    DfCvGraph[metric_name] = DfCvGraph["model"].map(values_by_model)

DfCvGraph

In [ ]:
def get_all_cv_history(base_path: Path) -> pd.DataFrame:
    """
    Parcourt tous les logs et extrait TOUTES les valeurs par epoch.
    Retourne un DataFrame: [base_model, epoch, metric_cv, value_cv]
    """
    records = []
    
    for model_dir in os.listdir(base_path):
        model_path = base_path / model_dir / "logs" / "scalars"
        if not model_path.is_dir():
            continue
            
        # On suppose que le nom du dossier est le 'base_name' (ex: Unet_bce)
        # Si le dossier a un suffixe numérique (ex: Unet_bce_run1), on nettoie si besoin.
        base_model = model_dir 
        
        for file in os.listdir(model_path):
            if not file.endswith(".csv") or not file.startswith("val_"):
                continue
                
            metric_name = file.replace("val_", "").replace(".csv", "")
            fp = model_path / file
            try:
                df = pd.read_csv(fp)
                if "value" not in df.columns or "step" not in df.columns or df.empty:
                    continue
                
                # On assume que step correspond grosso modo à l'epoch ou est proportionnel
                # Si 'step' est le global_step, il faut le convertir en epoch si nécessaire.
                # Ici on va assumer 1 ligne = 1 validation = 1 epoch pour simplifier, 
                # ou utiliser l'index + 1 comme epoch si le CSV est séquentiel.
                
                # Option A : Utiliser l'index comme proxy de l'epoch (souvent le cas dans les exports TB simples)
                for idx, row in df.iterrows():
                    records.append({
                        "base_model": base_model,
                        "epoch": idx + 1, # 1-based index
                        "metric_cv": metric_name,
                        "value_cv": row["value"]
                    })
                    
            except Exception as e:
                print(f"Skipping {fp}: {e}")

    return pd.DataFrame(records)
df_cv_history = get_all_cv_history(PATH_TB_LOGS)
df_cv_history

In [ ]:
df_graph_history = DfMeanErr.copy()
df_graph_history["base_model"] = df_graph_history["model"].map(base_name)
df_graph_history["epoch"] = df_graph_history["model"].map(extract_epochs)

df_graph_history = df_graph_history.dropna(subset=["epoch"]).sort_values(["base_model", "epoch"])

df_graph_clean = df_graph_history[["base_model", "epoch", "abs_error_mean", "metric"]].rename(columns={"metric": "metric_graph", "abs_error_mean": "value_graph"})

print(f"Graph data range: Epochs {df_graph_clean['epoch'].min()} to {df_graph_clean['epoch'].max()}")
df_graph_clean.sort_values(["base_model", "epoch"])

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from scipy import stats
from sklearn.metrics import r2_score

# --- 1. Alignement des données ---

# On renomme pour faciliter la fusion
df_cv_aligned = df_cv_history.rename(columns={"value_cv": "val_cv", "metric_cv": "metric_cv"})
df_graph_aligned = df_graph_clean.rename(columns={"value_graph": "val_graph", "metric_graph": "metric_graph"})

# Fusion sur base_model et epoch
# On utilise inner join car on a besoin des deux valeurs pour calculer une corrélation
df_merged = pd.merge(
    df_cv_aligned, 
    df_graph_aligned, 
    on=["base_model", "epoch"], 
    how="inner"
)

print(f"Données alignées : {len(df_merged)} paires de points trouvées.")
print("Exemple de données :")
display(df_merged.head())

# Vérification de la distribution des epochs
print(f"Plage d'epochs disponible : {df_merged['epoch'].min()} à {df_merged['epoch'].max()}")

In [ ]:
df_merged

In [ ]:
# mean value of val graph for base_model = 'Segformer_bce_dice' metric cv is specificity and metric graph is Convex_Area_Hull and epoch is 80
mean_value = df_merged[
    (df_merged['base_model'] == 'Segformer_bce_dice') & 
    (df_merged['metric_cv'] == 'Specificity') & 
    (df_merged['metric_graph'] == 'Convex_Area_Hull') & 
    (df_merged['epoch'] == 80)
]['val_graph'].mean()
print(f"Mean value of val_graph for Segformer_bce_dice, specificity, Convex_Area_Hull at epoch 80: {mean_value}")

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import tqdm
import warnings

# --- ÉTAPE 1 : Préparation de DfCvGraph pour l'analyse ---

def prepare_dfcv_for_correlation(df_input):
    df = df_input.copy()
    
    # 1. Extraction Base Model et Epoch depuis la colonne 'model' (ex: Unet_bce_100)
    # On suppose le format "Nom_Epoch"
    df['epoch'] = df['model'].str.extract(r'_(\d+)$').astype(float)
    df['base_model'] = df['model'].str.replace(r'_\d+$', '', regex=True)
    
    # 2. Identification des colonnes CV (tout ce qui n'est pas metadata ou metric graph)
    # Ajustez cette liste si vous avez d'autres colonnes de métadonnées
    metadata_cols = ['model', 'metric', 'abs_error_mean', 'std_abs_error', 
                     'epoch', 'base_model', 'root_ids', 'split', 'box', 'time', 
                     'train_epoch_loss', 'val_epoch_loss']
    
    # Les colonnes restantes sont supposées être les métriques CV (Precision, Recall, etc.)
    cv_metrics_cols = [c for c in df.columns if c not in metadata_cols]
    
    print(f"Métriques CV détectées : {cv_metrics_cols}")
    
    # 3. Transformation en format LONG (Melt)
    # On veut : [base_model, epoch, metric_graph, val_graph, metric_cv, val_cv]
    df_long = df.melt(
        id_vars=['base_model', 'epoch', 'metric', 'abs_error_mean'], # Ce qu'on garde fixe
        value_vars=cv_metrics_cols,                                  # Ce qu'on "déplie"
        var_name='metric_cv',
        value_name='val_cv'
    )
    
    # 4. Renommage pour correspondre à la fonction
    df_long = df_long.rename(columns={
        'metric': 'metric_graph',
        'abs_error_mean': 'val_graph'
    })
    
    # Nettoyage des NaNs (ex: si une métrique CV n'a pas été calculée)
    df_long = df_long.dropna(subset=['val_cv', 'val_graph', 'epoch'])
    
    return df_long

# Application de la préparation
df_ready = prepare_dfcv_for_correlation(DfCvGraph)

# --- ÉTAPE 2 : Fonction de calcul (Version Robuste) ---

def calculate_sliding_corr_robust(df_main, window_size=50, overlap=25):
    # Désactiver les warnings
    warnings.filterwarnings('ignore')
    
    # Définition des scénarios
    # Note: df_main a déjà 'base_model' grâce à l'étape 1
    scenarios = {
        "All Models": df_main,
        "Unet Only": df_main[df_main["base_model"].str.contains("Unet", case=False)],
        "Segformer Only": df_main[df_main["base_model"].str.contains("Segformer", case=False)],
    }
    
    # Ajout dynamique des familles de loss si détectées
    if df_main["base_model"].str.contains("bce", case=False).any():
        scenarios["BCE Loss Only"] = df_main[df_main["base_model"].str.contains("bce", case=False)]
    if df_main["base_model"].str.contains("dice", case=False).any():
        scenarios["Dice Loss Only"] = df_main[df_main["base_model"].str.contains("dice", case=False)]

    all_results = []
    step = window_size - overlap
    
    pbar = tqdm.tqdm(scenarios.items(), desc="Scenarios")
    
    for scenario_name, df_sub in pbar:
        if df_sub.empty:
            continue
            
        min_ep = df_sub["epoch"].min()
        max_ep = df_sub["epoch"].max()
        
        # Filtre : on ne garde que les métriques qui ont assez de données
        unique_cv = [m for m in df_sub["metric_cv"].unique() if len(df_sub[df_sub["metric_cv"]==m]) > 10]
        unique_graph = df_sub["metric_graph"].unique()
        
        # --- Pré-calcul des stats globales (Optimisation) ---
        global_stats = {}
        for m_cv in unique_cv:
            for m_graph in unique_graph:
                d_global = df_sub[
                    (df_sub["metric_cv"] == m_cv) & 
                    (df_sub["metric_graph"] == m_graph)
                ].dropna(subset=["val_cv", "val_graph"])
                
                # Check variance pour éviter RankWarning
                if len(d_global) > 5 and d_global["val_cv"].std() > 1e-9:
                    x = d_global["val_cv"].values
                    y = d_global["val_graph"].values
                    
                    try:
                        # Spearman Global
                        rho_g, _ = stats.spearmanr(x, y)
                        # R2 Global (via Linregress = plus stable)
                        slope, intercept, r_val, _, _ = stats.linregress(x, y)
                        r2_g = r_val**2
                    except:
                        rho_g, r2_g = np.nan, np.nan
                else:
                    rho_g, r2_g = np.nan, np.nan
                
                global_stats[(m_cv, m_graph)] = (rho_g, r2_g)

        # --- Boucle Glissante ---
        for start_ep in range(int(min_ep), int(max_ep), step):
            end_ep = start_ep + window_size
            mid_ep = start_ep + (window_size / 2)
            
            mask_win = (df_sub["epoch"] >= start_ep) & (df_sub["epoch"] < end_ep)
            df_win = df_sub[mask_win]
            
            if len(df_win) < 5: 
                continue
                
            for m_cv in unique_cv:
                for m_graph in unique_graph:
                    d = df_win[
                        (df_win["metric_cv"] == m_cv) & 
                        (df_win["metric_graph"] == m_graph)
                    ].dropna(subset=["val_cv", "val_graph"])
                    
                    if len(d) < 5:
                        continue
                        
                    x = d["val_cv"].values
                    y = d["val_graph"].values
                    
                    # Vérification Variance Nulle locale
                    if np.std(x) < 1e-9 or np.std(y) < 1e-9:
                        rho, r2 = np.nan, np.nan
                    else:
                        try:
                            rho, _ = stats.spearmanr(x, y)
                            # Utilisation de linregress au lieu de polyfit pour éviter les warnings
                            slope, intercept, r_val, _, _ = stats.linregress(x, y)
                            r2 = r_val**2
                        except:
                            rho, r2 = np.nan, np.nan
                    
                    glob_rho, glob_r2 = global_stats.get((m_cv, m_graph), (np.nan, np.nan))

                    all_results.append({
                        "scenario": scenario_name,
                        "window_center": mid_ep,
                        "metric_cv": m_cv,
                        "metric_graph": m_graph,
                        "spearman": rho,
                        "r2": r2,
                        "global_spearman": glob_rho,
                        "global_r2": glob_r2,
                        "count": len(d)
                    })
    
    warnings.resetwarnings()
    return pd.DataFrame(all_results)


df_sliding_scenarios = calculate_sliding_corr_robust(df_ready, window_size=50, overlap=25)

print("Aperçu des résultats :")
display(df_sliding_scenarios.head())

In [ ]:
df_sliding_scenarios

In [ ]:
def plot_scenario_comparison(metric_cv, metric_graph):
    """
    Affiche l'évolution du Spearman pour une paire de métriques,
    en comparant les différents scénarios (All, Unet, Segformer).
    """
    df_plot = df_sliding_scenarios[
        (df_sliding_scenarios["metric_cv"] == metric_cv) & 
        (df_sliding_scenarios["metric_graph"] == metric_graph)
    ]
    
    if df_plot.empty:
        print("Pas de données pour cette combinaison.")
        return
        
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Palette de couleurs pour les scénarios
    scenarios = df_plot["scenario"].unique()
    colors = {"All Models": "grey", "Unet Only": "tab:blue", "Segformer Only": "tab:orange"}
    
    for sc in scenarios:
        d = df_plot[df_plot["scenario"] == sc].sort_values("window_center")
        style = "--" if sc == "All Models" else "-"
        width = 2 if sc == "All Models" else 3
        
        ax.plot(d["window_center"], d["spearman"], 
                label=sc, linestyle=style, linewidth=width, marker='o')
        
    # write global spearman as horizontal lines
    for sc in scenarios:
        d = df_plot[df_plot["scenario"] == sc]
        if not d.empty:
            glob_rho = d["global_spearman"].iloc[0]
            glob_r2 = d["global_r2"].iloc[0]
            c_rho = colors.get(sc, "black")
            c_r2 = colors.get(sc, "grey")
            ax.axhline(glob_rho, color=c_rho, linestyle=":", linewidth=1.5, alpha=0.7)
            ax.text(df_plot["window_center"].max(), glob_rho, f" {sc} Global ρ={glob_rho:.2f}", color=c_rho, va="center", fontsize=9)
            ax.axhline(glob_r2, color=c_r2, linestyle="--", linewidth=1.5, alpha=0.7)
            ax.text(df_plot["window_center"].max(), glob_r2, f" {sc} Global R²={glob_r2:.2f}", color=c_r2, va="center", fontsize=9)
                
    ax.set_title(f"Dynamic Correlation: {metric_cv} vs {metric_graph}")
    ax.set_ylabel("Spearman Correlation")
    ax.set_xlabel("Training Epoch")
    ax.axhline(0, color='black', alpha=0.3, linestyle=':')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

# Widgets
w_cv = widgets.Dropdown(options=sorted(df_sliding_scenarios["metric_cv"].unique()), description="CV Metric")
w_gr = widgets.Dropdown(options=sorted(df_sliding_scenarios["metric_graph"].unique()), description="Graph Metric")

widgets.interactive(plot_scenario_comparison, metric_cv=w_cv, metric_graph=w_gr)

In [ ]:
import pandas as pd
import numpy as np
import re
from scipy import stats
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import ipywidgets as widgets
import seaborn as sns

# --- 1. Nettoyage et Enrichissement de DfCvGraph2 ---
# On travaille sur une copie pour ne pas toucher l'original
# Remplacez DfCvGraph2 par DfCvGraph si vous voulez tout le monde
df_work = DfCvGraph.copy() 

def extract_metadata(row):
    model_name = str(row['model'])
    
    # 1. Extraction Epoch (ex: ..._120)
    m_ep = re.search(r'_(\d{2,4})(?:\D|$)', model_name)
    epoch = int(m_ep.group(1)) if m_ep else np.nan
    
    # 2. Extraction Architecture (Unet / Segformer)
    if "segformer" in model_name.lower():
        arch = "Segformer"
    elif "unet" in model_name.lower():
        arch = "Unet"
    else:
        arch = "Other"
        
    # 3. Extraction Loss (bce, dice, etc.)
    # On suppose format: Arch_LOSS_epoch
    # On retire l'arch et l'epoch pour garder le milieu
    clean_name = model_name.lower().replace(arch.lower() + "_", "")
    if m_ep:
        clean_name = clean_name.replace(f"_{m_ep.group(1)}", "")
    loss = clean_name
    
    return pd.Series([epoch, arch, loss], index=['epoch', 'arch', 'loss'])

# Application
meta = df_work.apply(extract_metadata, axis=1)
df_work = pd.concat([df_work, meta], axis=1)

# On filtre les epochs non trouvées (-1 ou NaN)
df_work = df_work.dropna(subset=['epoch']).sort_values(['arch', 'loss', 'epoch'])

print("Colonnes disponibles pour CV:", df_work.columns)
display(df_work[['model', 'arch', 'loss', 'epoch']].head())

In [ ]:
df_work

In [ ]:
def calculate_sliding_corr_from_dfcv(df, cv_cols, window_size=50, overlap=25):
    results = []
    step = window_size - overlap
    
    # Liste des métriques graphiques (ex: Hausdorff, APLS...)
    graph_metrics = df["metric"].unique()
    
    # Définition des groupes (Scénarios)
    # On groupe par (Architecture + Loss) pour avoir des séries temporelles cohérentes
    # Ex: Scenario = "Unet - bce"
    grouped = df.groupby(['arch', 'loss'])
    
    for (arch, loss_name), group_df in grouped:
        scenario_name = f"{arch} - {loss_name}"
        
        # Bornes temporelles du groupe
        min_ep, max_ep = group_df['epoch'].min(), group_df['epoch'].max()
        
        # Pour chaque métrique Graphique (ex: Hausdorff)
        for g_metric in graph_metrics:
            sub_g = group_df[group_df["metric"] == g_metric]
            
            if len(sub_g) < 5: continue
            
            # Pour chaque métrique CV (ex: Precision, val_loss...)
            for cv_col in cv_cols:
                if cv_col not in sub_g.columns: continue
                
                # Données brutes (X=CV, Y=GraphError)
                data_pair = sub_g[[cv_col, "abs_error_mean", "epoch"]].dropna()
                
                # --- Calcul GLOBAL (pour ligne horizontale) ---
                try:
                    glob_rho, _ = stats.spearmanr(data_pair[cv_col], data_pair["abs_error_mean"])
                    
                    if len(data_pair) > 1:
                        z = np.polyfit(data_pair[cv_col], data_pair["abs_error_mean"], 1)
                        glob_r2 = r2_score(data_pair["abs_error_mean"], np.poly1d(z)(data_pair[cv_col]))
                    else: glob_r2 = 0
                except:
                    glob_rho, glob_r2 = np.nan, np.nan

                # --- Calcul GLISSANT ---
                for start_ep in range(int(min_ep), int(max_ep), step):
                    end_ep = start_ep + window_size
                    mid_ep = start_ep + (window_size/2)
                    
                    win = data_pair[(data_pair["epoch"] >= start_ep) & (data_pair["epoch"] < end_ep)]
                    if len(win) < 4: continue
                    
                    x = win[cv_col].values
                    y = win["abs_error_mean"].values
                    
                    # Spearman
                    try: rho, _ = stats.spearmanr(x, y)
                    except: rho = np.nan
                    
                    # R2
                    try:
                        if len(np.unique(x)) > 1:
                            z_win = np.polyfit(x, y, 1)
                            r2 = r2_score(y, np.poly1d(z_win)(x))
                        else: r2 = 0
                    except: r2 = np.nan
                    
                    results.append({
                        "scenario": scenario_name,
                        "arch": arch,
                        "loss_group": loss_name,
                        "window_center": mid_ep,
                        "metric_graph": g_metric, # Target
                        "metric_cv": cv_col,      # Input
                        "spearman": rho,
                        "r2": r2,
                        "global_spearman": glob_rho,
                        "global_r2": glob_r2
                    })
                    
    return pd.DataFrame(results)

# --- Configuration des colonnes CV à analyser ---
# Ajoutez ici toutes les colonnes présentes dans DfCvGraph que vous voulez tester
cv_columns_to_test = [
    "Precision", "Recall", "F1Score", "MeanIoU", 
    "train_epoch_loss", "val_epoch_loss"
    # Ajoutez vos colonnes de loss spécifiques si elles existent (ex: "loss_BCE")
] 
# Filtrer celles qui existent vraiment
cv_columns_to_test = [c for c in cv_columns_to_test if c in df_work.columns]

df_sliding_v2 = calculate_sliding_corr_from_dfcv(df_work, cv_columns_to_test, window_size=140, overlap=80)


In [ ]:
df_sliding_v2

In [ ]:
def plot_scenario_comparison_v2(metric_cv, metric_graph, selected_scenarios, show_r2):
    """
    Plot dynamique comparant les scénarios sélectionnés depuis DfCvGraph.
    """
    if not selected_scenarios:
        print("Veuillez sélectionner au moins un scénario.")
        return

    # Filtrage
    df_plot = df_sliding_v2[
        (df_sliding_v2["metric_cv"] == metric_cv) & 
        (df_sliding_v2["metric_graph"] == metric_graph) &
        (df_sliding_v2["scenario"].isin(selected_scenarios))
    ]
    
    if df_plot.empty:
        print("Pas de données pour cette combinaison.")
        return
        
    fig, ax = plt.subplots(figsize=(16, 9))
    
    # Couleurs distinctes
    unique_scens = sorted(list(selected_scenarios))
    cmap = plt.cm.get_cmap('tab10', len(unique_scens)) 
    colors = {sc: cmap(i) for i, sc in enumerate(unique_scens)}
    
    # Plot
    for sc in unique_scens:
        d = df_plot[df_plot["scenario"] == sc].sort_values("window_center")
        if d.empty: continue
        
        c = colors[sc]
        
        # Courbe principale (Spearman ou R2 selon choix)
        y_val = d["r2"] if show_r2 else d["spearman"]
        label_metric = "R²" if show_r2 else "Spearman"
        
        # Courbe lissée
        ax.plot(d["window_center"], y_val, 
                label=f"{sc}", color=c, linewidth=2.5, marker='o', markersize=6, alpha=0.8)
        
        # Ligne globale (moyenne sur tout l'entrainement)
        glob_val = d["global_r2"].iloc[0] if show_r2 else d["global_spearman"].iloc[0]
        ax.axhline(glob_val, color=c, linestyle=":", linewidth=1.5, alpha=0.5)
        
        # Annotation en fin de courbe
        last_x = d["window_center"].max()
        ax.text(last_x + 2, glob_val, f"Global: {glob_val:.2f}", color=c, fontsize=9, va='center')

    ylabel = "Linear Fit (R²)" if show_r2 else "Rank Correlation (Spearman)"
    ax.set_title(f"Dynamic {ylabel}: {metric_cv} vs {metric_graph}")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Epoch (Window Center)")
    
    # Bornes Y
    if not show_r2: # Spearman [-1, 1]
        ax.set_ylim(-1.1, 1.1)
        ax.axhline(0, color='black', alpha=0.8, linewidth=1)
    else: # R2 [open, 1]
        ax.set_ylim(bottom=max(-1, df_plot["r2"].min() - 0.1), top=1.05)
        ax.axhline(0, color='black', alpha=0.8, linewidth=1)

    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(title="Model & Loss Scenario", loc='upper left', bbox_to_anchor=(1, 1))
    
    plt.tight_layout()
    plt.show()

# --- Widgets ---

avail_scenarios = sorted(df_sliding_v2['scenario'].unique().tolist())
avail_cv = sorted(df_sliding_v2['metric_cv'].unique().tolist())
avail_gr = sorted(df_sliding_v2['metric_graph'].unique().tolist())

w_cv = widgets.Dropdown(options=avail_cv, value=avail_cv[0], description="Metric CV")
w_gr = widgets.Dropdown(options=avail_gr, value=avail_gr[0], description="Metric Graph")

# SelectMultiple: Ctrl+Click pour sélectionner plusieurs
w_scen = widgets.SelectMultiple(
    options=avail_scenarios,
    value=[avail_scenarios[0]], 
    description="Scenarios",
    rows=min(10, len(avail_scenarios))
)

w_type = widgets.Checkbox(value=False, description="Afficher R² (sinon Spearman)")

ui = widgets.VBox([
    widgets.HBox([w_cv, w_gr, w_type]),
    w_scen
])

out = widgets.interactive_output(
    plot_scenario_comparison_v2, 
    {'metric_cv': w_cv, 'metric_graph': w_gr, 'selected_scenarios': w_scen, 'show_r2': w_type}
)

display(ui, out)

In [ ]:
def plot_scenario_comparison_advanced(metric_cv, metric_graph, selected_scenarios, show_global_lines):
    """
    Plot dynamique comparant les scénarios sélectionnés.
    """
    if not selected_scenarios:
        print("Veuillez sélectionner au moins un scénario.")
        return

    # Filtrage
    df_plot = df_sliding_scenarios[
        (df_sliding_scenarios["metric_cv"] == metric_cv) & 
        (df_sliding_scenarios["metric_graph"] == metric_graph) &
        (df_sliding_scenarios["scenario"].isin(selected_scenarios))
    ]
    
    if df_plot.empty:
        print("Pas de données pour cette combinaison.")
        return
        
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # Palette dynamique
    unique_scens = sorted(list(selected_scenarios))
    # Utilisation d'une colormap qualitative pour bien distinguer les lignes
    cmap = plt.cm.get_cmap('tab10', len(unique_scens)) 
    colors = {sc: cmap(i) for i, sc in enumerate(unique_scens)}
    
    # 1. Plot des courbes (Sliding Window)
    for sc in unique_scens:
        d = df_plot[df_plot["scenario"] == sc].sort_values("window_center")
        if d.empty: continue
        
        c = colors[sc]
        ax.plot(d["window_center"], d["spearman"], 
                label=f"{sc} (Rolling)", color=c, linewidth=2.5, marker='o', markersize=5, alpha=0.8)
        
    # 2. Plot des lignes globales (Optionnel)
    if show_global_lines:
        x_max = df_plot["window_center"].max()
        x_min = df_plot["window_center"].min()
        
        for sc in unique_scens:
            d = df_plot[df_plot["scenario"] == sc]
            if d.empty: continue
            
            # On prend la valeur de la première ligne (c'est la même partout pour un scénario donné)
            glob_rho = d["global_spearman"].iloc[0]
            
            c = colors[sc]
            # Ligne pointillée
            ax.axhline(glob_rho, color=c, linestyle=":", linewidth=1.5, alpha=0.6)
            
            # Annotation textuelle sur le côté droit
            ax.text(x_max, glob_rho, f" Global ρ={glob_rho:.2f}", 
                    color=c, va="center", ha="left", fontsize=9, fontweight='bold')

    ax.set_title(f"Correlation Dynamics: {metric_cv} vs {metric_graph}")
    ax.set_ylabel("Spearman Correlation")
    ax.set_xlabel("Training Epoch")
    ax.set_ylim(-1.1, 1.1)
    
    # Lignes guides
    ax.axhline(0, color='black', alpha=0.8, linewidth=1)
    ax.axhline(0.5, color='gray', alpha=0.3, linestyle='--')
    ax.axhline(-0.5, color='gray', alpha=0.3, linestyle='--')
    
    ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

# --- Widgets ---

all_scenarios = sorted(df_sliding_scenarios['scenario'].unique().tolist())
all_cv = sorted(df_sliding_scenarios['metric_cv'].unique().tolist())
all_gr = sorted(df_sliding_scenarios['metric_graph'].unique().tolist())

w_cv = widgets.Dropdown(options=all_cv, value=all_cv[0], description="CV Metric")
w_gr = widgets.Dropdown(options=all_gr, value=all_gr[0], description="Graph Metric")

# SelectMultiple permet de choisir plusieurs items (Ctrl+Click / Cmd+Click)
w_scenarios = widgets.SelectMultiple(
    options=all_scenarios,
    value=[all_scenarios[0]], # Valeur par défaut
    description="Scenarios",
    rows=len(all_scenarios)
)

w_global = widgets.Checkbox(value=True, description="Show Global Lines")

widgets.interactive(
    plot_scenario_comparison_advanced, 
    metric_cv=w_cv, 
    metric_graph=w_gr, 
    selected_scenarios=w_scenarios,
    show_global_lines=w_global
)